# RSNA Knee — CNX-r7 IN-22k v3b — T4 x2 — TTA x3

Notebook **self-contained** (internet off, GPU **T4 x2**). **CNX seul, TTA x3.**

- DICOM test → volumes crop **130 mm**, bande 20–80 %, (6, 24, 224, 224)
- protocole 6 slots FS/T1. Pas de flip de latéralité
- KneeNet **CNX-r7** : ConvNeXt-Small `convnext_small.fb_in22k_ft_in1k`, tête R7
  train `cnx_r7_in22k_v3b_oofmask_full` **SWA** epochs 7, 8, 9 et 10
  (gold in-sample moyen 0.976)
- poids `cnx_r7_swa_ep7_10.pt`, dataset `sultent/rsna-knee-cnx-r7-swa78910`
- **TTA x3**, sans l'identite : hflip, center-crop zoom 1.05, center-crop zoom 1.10
- echange med/lat seulement apres le hflip. Pas de rotation. Pas d'ensemble
- etudes reparties sur cuda:0 et cuda:1
- écrit `/kaggle/working/submission.csv`


In [ ]:
import os
from pathlib import Path

KAGGLE = os.path.exists('/kaggle')
if KAGGLE:
    DATA_DIR = Path('/kaggle/input')
    print('contenu /kaggle/input :', sorted(p.name for p in DATA_DIR.iterdir()))

    COMP_DIR = None
    for base in [DATA_DIR / 'competitions', DATA_DIR]:
        if base.exists():
            for p in sorted(base.iterdir()):
                if (p / 'test.csv').exists():
                    COMP_DIR = p
                    break
        if COMP_DIR is not None:
            break
    if COMP_DIR is None:
        raise FileNotFoundError('test.csv introuvable sous /kaggle/input')

    search_roots = []
    if (DATA_DIR / 'datasets').exists():
        search_roots.append(DATA_DIR / 'datasets')
    search_roots.append(DATA_DIR)

    def _find_weight(names):
        for name in names:
            for root in search_roots:
                hits = list(root.rglob(name))
                if hits:
                    return hits[0]
        raise FileNotFoundError(f'{names} introuvable (dataset weights attache ?)')

    WEIGHTS_PATH = _find_weight(('cnx_r7_swa_ep7_10.pt',))
else:
    COMP_DIR = Path('../data/csv')
    WEIGHTS_PATH = Path('kaggle_dataset_cnx/_upload_swa78910/cnx_r7_swa_ep7_10.pt')

TEST_CSV = COMP_DIR / 'test.csv'
TEST_SERIES_CSV = COMP_DIR / 'test_series.csv'
TEST_SERIES_DIR = COMP_DIR / 'test_series'
SAMPLE_SUB = COMP_DIR / 'sample_submission.csv'

import torch

def _pick_device():
    if not torch.cuda.is_available():
        return 'cpu'
    try:
        m = torch.nn.Conv2d(3, 8, 3, padding=1).cuda()
        _ = m(torch.zeros(1, 3, 32, 32, device='cuda')).sum().item()
        return 'cuda'
    except Exception as e:
        print('CUDA inutilisable, repli CPU :', str(e)[:120])
        return 'cpu'

DEVICE = _pick_device()
if DEVICE == 'cuda':
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
try:
    torch.set_float32_matmul_precision('high')
except Exception:
    pass

print('competition dir :', COMP_DIR)
print('poids            :', WEIGHTS_PATH, '| existe :', WEIGHTS_PATH.exists())
print('device           :', DEVICE)


In [ ]:
# -*- coding: utf-8 -*-
"""Pretraitement DICOM -> volume uint8, aligne sur la conversion NPZ.

Recette identique a `kaggle_DICOM_exctract/_gen_convert_nb.py` (grayscale)
et `_gen_convert_rgb25d_nb.py` (voisins DICOM) :
1. Lecture des .dcm, tri InstanceNumber (fallback : normale IOP).
2. RescaleSlope/Intercept, inversion MONOCHROME1.
3. Crop centre 130 mm (PixelSpacing) sinon centre 70 %.
4. Percentiles 1-99 -> [0, 1].
5. Bande profondeur 20-80 % (evite air / bords), puis linspace 24 coupes.
6. Resize IMG_SIZE (224 par defaut, 320 pour crop130_320), uint8.
7. Si rgb_neighbors : pour chaque centre k du linspace, stocker k-1, k, k+1
   dans le volume ORIGINAL (1 coupe DICOM d'ecart), pas dans les 24.
"""
from __future__ import annotations

from pathlib import Path

import cv2
import numpy as np

PROTOCOL_SLOTS: list[tuple[str, int]] = [
    ("Sagittal", 1),
    ("Coronal", 1),
    ("Axial", 1),
    ("Sagittal", 0),
    ("Coronal", 0),
    ("Axial", 0),
]

def laterality_from_study_dir(study_dir):
    raise RuntimeError('laterality off sur ce kernel')

def normalise_laterality(vol, plane, lat):
    raise RuntimeError('laterality off sur ce kernel')


N_SLICES = 24
IMG_SIZE = 224
CROP_MM = 130.0
CROP_REL = 0.70
BAND = (0.20, 0.80)


def _slice_sort_key(ds) -> float:
    """Cle de tri : InstanceNumber, sinon projection sur la normale."""
    inst = getattr(ds, "InstanceNumber", None)
    if inst is not None:
        try:
            return float(inst)
        except (TypeError, ValueError):
            pass
    ipp = getattr(ds, "ImagePositionPatient", None)
    iop = getattr(ds, "ImageOrientationPatient", None)
    if ipp is not None and iop is not None:
        row = np.array(iop[:3], dtype=np.float64)
        col = np.array(iop[3:], dtype=np.float64)
        normal = np.cross(row, col)
        return float(np.dot(np.array(ipp, dtype=np.float64), normal))
    if ipp is not None:
        return float(ipp[2])
    return 0.0


def _spacing_mm(ds):
    ps = getattr(ds, "PixelSpacing", None) or getattr(ds, "ImagerPixelSpacing", None)
    if ps is None:
        return None
    try:
        return float(ps[0]), float(ps[1])
    except (TypeError, ValueError, IndexError):
        return None


def _center_crop(img: np.ndarray, row_mm, col_mm) -> np.ndarray:
    h, w = img.shape[:2]
    if row_mm is not None and col_mm is not None and row_mm > 0 and col_mm > 0:
        ch = int(round(CROP_MM / row_mm))
        cw = int(round(CROP_MM / col_mm))
        ch = min(h, max(8, ch))
        cw = min(w, max(8, cw))
    else:
        ch = max(8, int(round(h * CROP_REL)))
        cw = max(8, int(round(w * CROP_REL)))
        ch, cw = min(h, ch), min(w, cw)
    y0 = (h - ch) // 2
    x0 = (w - cw) // 2
    return img[y0:y0 + ch, x0:x0 + cw]


def _rescale(ds, arr: np.ndarray) -> np.ndarray:
    arr = arr.astype(np.float32, copy=False)
    slope = float(getattr(ds, "RescaleSlope", 1.0) or 1.0)
    intercept = float(getattr(ds, "RescaleIntercept", 0.0) or 0.0)
    arr = arr * slope + intercept
    if getattr(ds, "PhotometricInterpretation", "MONOCHROME2") == "MONOCHROME1":
        arr = arr.max() - arr
    return arr


def _to_u8(sl: np.ndarray, img_size: int = IMG_SIZE) -> np.ndarray:
    sl = cv2.resize(sl, (img_size, img_size), interpolation=cv2.INTER_AREA)
    return np.clip(sl * 255.0, 0, 255).round().astype(np.uint8)


def load_series_dicom(
    series_dir: str | Path,
    rgb_neighbors: bool = False,
    img_size: int = IMG_SIZE,
) -> np.ndarray:
    """Un dossier de .dcm -> (24, S, S) ou (24, 3, S, S) uint8, S=`img_size`.

    rgb_neighbors=False : grayscale crop130 (linspace 24).
    rgb_neighbors=True  : R=k-1, G=k, B=k+1 dans le volume original.
    """
    import pydicom

    series_dir = Path(series_dir)
    files = sorted(series_dir.glob("*.dcm"))
    if not files:
        raise FileNotFoundError(f"pas de .dcm dans {series_dir}")

    datasets = []
    for f in files:
        try:
            datasets.append(pydicom.dcmread(str(f)))
        except Exception:
            continue
    if not datasets:
        raise RuntimeError(f"aucun DICOM lisible dans {series_dir}")
    datasets.sort(key=_slice_sort_key)

    frames = []
    sp = None
    for ds in datasets:
        arr = ds.pixel_array
        if sp is None:
            sp = _spacing_mm(ds)
        arr = _rescale(ds, arr)
        if arr.ndim == 2:
            frames.append(arr)
        elif arr.ndim == 3:
            frames.extend([arr[i] for i in range(arr.shape[0])])
        else:
            raise RuntimeError(f"ndim inattendu {arr.ndim}")
    if not frames:
        raise RuntimeError("aucune coupe")

    row_mm, col_mm = (sp if sp is not None else (None, None))
    cropped = [_center_crop(im, row_mm, col_mm) for im in frames]
    h0, w0 = cropped[0].shape[:2]
    stacked = []
    for c in cropped:
        if c.shape[:2] != (h0, w0):
            c = cv2.resize(c, (w0, h0), interpolation=cv2.INTER_AREA)
        stacked.append(c)
    vol = np.stack(stacked, axis=0)

    lo, hi = np.percentile(vol, 1.0), np.percentile(vol, 99.0)
    if hi <= lo:
        hi = lo + 1.0
    vol = np.clip((vol - lo) / (hi - lo), 0, 1)

    n = vol.shape[0]
    a = int(np.floor(BAND[0] * (n - 1)))
    b = int(np.ceil(BAND[1] * (n - 1)))
    if b - a + 1 < 8:
        a, b = 0, n - 1
    idx_band = np.linspace(0, b - a, N_SLICES).round().astype(int)
    idx_orig = np.clip(a + idx_band, 0, n - 1)

    size = int(img_size)
    if not rgb_neighbors:
        out = np.empty((N_SLICES, size, size), dtype=np.uint8)
        for i, k in enumerate(idx_orig):
            out[i] = _to_u8(vol[int(k)], size)
        return out

    out = np.empty((N_SLICES, 3, size, size), dtype=np.uint8)
    last = n - 1
    for i, k in enumerate(idx_orig):
        k = int(k)
        km = 0 if k <= 0 else k - 1
        kp = last if k >= last else k + 1
        out[i, 0] = _to_u8(vol[km], size)
        out[i, 1] = _to_u8(vol[k], size)
        out[i, 2] = _to_u8(vol[kp], size)
    return out


def _pick_plane_row(series_rows, plane: str):
    """Fluid-sensitive d'abord, sinon n'importe quelle serie du plan."""
    rows = series_rows[series_rows.Anatomical_Plane == plane]
    if len(rows) == 0:
        return None
    fluid_col = "Fluid_Sensitive" if "Fluid_Sensitive" in rows.columns else None
    fluid = rows[rows[fluid_col] == 1] if fluid_col else rows.iloc[0:0]
    if len(fluid) == 0 and "Fat_Suppression" in rows.columns:
        fluid = rows[rows.Fat_Suppression == 1]
    pick = (fluid if len(fluid) else rows).sort_values("SeriesInstanceUID").iloc[0]
    return pick


def _pick_nonfluid_row(series_rows, exclude: set[str]):
    """T1 / non-fluide pas deja utilisee. Sagittal d'abord."""
    for plane in ("Sagittal", "Coronal", "Axial"):
        rows = series_rows[series_rows.Anatomical_Plane == plane]
        if "Fluid_Sensitive" in rows.columns:
            rows = rows[rows.Fluid_Sensitive == 0]
        rows = rows[~rows.SeriesInstanceUID.isin(exclude)]
        if len(rows) == 0:
            continue
        return rows.sort_values("SeriesInstanceUID").iloc[0]
    return None


def _pick_protocol_row(series_rows, plane: str, fluid: int):
    """Un contraste exact (FS ou T1) sur un plan. Pas de repli."""
    rows = series_rows[series_rows.Anatomical_Plane == plane]
    if len(rows) == 0:
        return None
    if "Fluid_Sensitive" in rows.columns:
        rows = rows[rows.Fluid_Sensitive == int(fluid)]
    elif fluid == 1 and "Fat_Suppression" in rows.columns:
        rows = rows[rows.Fat_Suppression == 1]
    elif fluid == 0 and "Fat_Suppression" in rows.columns:
        rows = rows[rows.Fat_Suppression == 0]
    if len(rows) == 0:
        return None
    return rows.sort_values("SeriesInstanceUID").iloc[0]


def study_to_tensor(study_dir: str | Path, series_rows,
                    n_series: int = 3,
                    rgb_neighbors: bool = False,
                    img_size: int = IMG_SIZE,
                    laterality: bool = False) -> np.ndarray:
    """Une etude -> float32 dans [0, 1].

    Grayscale : (n_series, 24, S, S) avec S=`img_size`.
    rgb_neighbors : (n_series, 24, 3, S, S) R=k-1 G=k B=k+1 DICOM.

    series_rows : DataFrame des series de l'etude (colonnes SeriesInstanceUID,
    Anatomical_Plane, Fluid_Sensitive, Fat_Suppression).
    n_series=3 : sag / cor / ax, fluid-sensitive d'abord.
    n_series=4 : + T1 extra (sagittal prioritaire).
    n_series=6 : 3 FS + 3 T1 par plan, sans repli (v20b).
    laterality=True : canon gauche (flip R), meme recette que le train.
    """
    planes = ("Sagittal", "Coronal", "Axial")
    study_dir = Path(study_dir)
    size = int(img_size)
    if rgb_neighbors:
        out = np.zeros(
            (n_series, N_SLICES, 3, size, size), dtype=np.float32,
        )
    else:
        out = np.zeros((n_series, N_SLICES, size, size), dtype=np.float32)
    used: set[str] = set()
    lat = laterality_from_study_dir(study_dir) if laterality else None

    def _fill(i: int, pick, plane: str | None) -> None:
        if pick is None:
            return
        series_dir = study_dir / pick.SeriesInstanceUID
        if not series_dir.exists():
            return
        try:
            vol = load_series_dicom(
                series_dir, rgb_neighbors=rgb_neighbors, img_size=size)
        except Exception as e:
            print(f"  [warn] serie {pick.SeriesInstanceUID}: {e}")
            return
        used.add(str(pick.SeriesInstanceUID))
        vol = vol.astype(np.float32) / 255.0
        if laterality:
            pl = plane or getattr(pick, "Anatomical_Plane", None)
            vol = normalise_laterality(vol, pl, lat)
        out[i] = vol

    if n_series == 6:
        for i, (plane, fluid) in enumerate(PROTOCOL_SLOTS):
            _fill(i, _pick_protocol_row(series_rows, plane, fluid), plane)
        return out

    for i, plane in enumerate(planes):
        _fill(i, _pick_plane_row(series_rows, plane), plane)
    if n_series >= 4:
        pick = _pick_nonfluid_row(series_rows, used)
        pl = None if pick is None else str(pick.Anatomical_Plane)
        _fill(3, pick, pl)
    return out


In [ ]:
# -*- coding: utf-8 -*-
"""KneeNet-CNX-r7 — tete R7 (Heliosli) sur l'entree hybrid025.

Meme tenseur que CNX-a : (B, 6, 24, 224, 224) grayscale [0, 1], slots
v20b. Le cache px384_v4 n'est pas reconstruit : c'est leur rendu
precalcule (384 px, vues 100 mm, meta DICOM). Ici chaque coupe valide
est une fenetre 2.5D (k-1, k, k+1), comme les 3 canaux de R7.

Graphe, repris de compact_train.Net (spatial + extent + syn_decomp) :
  - ConvNeXt-Small stride 32, drop_path 0.1, carte brute (sans head.norm).
  - Attention spatiale par pathologie (requete nulle a l'init = moyenne).
  - LayerNorm de tete apres le pooling, par classe.
  - Attention par pathologie sur les coupes du study.
  - Etendue OA : gamma * log1p(aire/100), gamma nul a l'init.
    Aire = produit sagittal x coronal (OA TF) ou axial x sagittal (PF).
    Epaisseur de dalle : 3.5 mm x nombre de coupes valides du slot
    (le zip crop130 n'a pas le spacing DICOM).
  - Synovite = part specifique + beta * stopgrad(logit epanchement).

Pas de tetes d'etat ni d'auxiliaire fastMRI : les cibles sont v3b (12
colonnes). Ces tetes ne changent pas les logits tant que leur porte
vaut 0, et il n'y a pas de labels d'etat pour les apprendre.
"""
from __future__ import annotations

import math
from pathlib import Path

import timm
print('timm', timm.__version__, flush=True)
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint

N_TARGETS = 12
TIMM_NAME = "convnext_small.fb_in22k_ft_in1k"
DEFAULT_WEIGHTS = "weights/convnext_small-fb_in22k_ft_in1k"
N_PROTOCOL_SLOTS = 6
HIDDEN = 384
OA_MED, OA_LAT, OA_PF = 4, 5, 6
SYN, EFF = 8, 7
# slots v20b : SAG_FS, COR_FS, AX_FS, SAG_T1, COR_T1, AX_T1
SLOT_PLANE = (0, 1, 2, 0, 1, 2)
SLOT_FLUID = (1, 1, 1, 0, 0, 0)
SLAB_MM = 3.5
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


def _weights_file(path: str | Path) -> Path:
    p = Path(path)
    if p.is_dir():
        for name in ("model.safetensors", "pytorch_model.bin"):
            cand = p / name
            if cand.is_file():
                return cand
        raise FileNotFoundError(
            f"aucun checkpoint dans {p.resolve()} "
            "(attendu model.safetensors)"
        )
    if not p.is_file():
        raise FileNotFoundError(f"poids ConvNeXt introuvables : {p.resolve()}")
    return p


class _ConvNextEncoder(nn.Module):
    """Carte stride 32 avant head.norm. drop_path 0.1, comme R7."""

    def __init__(self, name: str = TIMM_NAME,
                 weights_path: str = DEFAULT_WEIGHTS,
                 pretrained: bool = True) -> None:
        super().__init__()
        kwargs = dict(
            num_classes=0, global_pool="", output_stride=32,
            drop_path_rate=0.1,
        )
        if pretrained:
            ckpt = _weights_file(weights_path)
            self.backbone = timm.create_model(
                name, pretrained=True,
                pretrained_cfg_overlay={"file": str(ckpt)}, **kwargs,
            )
            print(f"ConvNeXt charge : {name} | {ckpt} | stride 32 | drop_path 0.1",
                  flush=True)
        else:
            self.backbone = timm.create_model(
                name.split(".", 1)[0], pretrained=False, **kwargs,
            )
        self.num_features = int(self.backbone.num_features)
        cfg = self.backbone.pretrained_cfg if pretrained else {}
        self.mean = tuple(float(v) for v in cfg.get("mean", IMAGENET_MEAN))
        self.std = tuple(float(v) for v in cfg.get("std", IMAGENET_STD))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.backbone.forward_features(x)


class KneeNet(nn.Module):
    arch_id = "kneenet_cnx_r7"

    def __init__(self, backbone: str = TIMM_NAME,
                 n_series: int = N_PROTOCOL_SLOTS,
                 n_targets: int = N_TARGETS, hidden: int = HIDDEN,
                 dropout: float = 0.1, pretrained: bool = True,
                 encode_chunk: int = 4, max_depth: int = 64,
                 backbone_weights: str = DEFAULT_WEIGHTS) -> None:
        super().__init__()
        self.n_series = n_series
        self.n_targets = n_targets
        self.encode_chunk = int(encode_chunk)
        self.backbone_name = str(backbone)
        self.hidden = int(hidden)
        self.max_depth = int(max_depth)

        self.encoder = _ConvNextEncoder(
            name=self.backbone_name, weights_path=backbone_weights,
            pretrained=pretrained,
        )
        d = self.encoder.num_features
        norm = self.encoder.backbone.head.norm
        if isinstance(norm, nn.Identity):
            raise RuntimeError("ConvNeXt sans head.norm : le pooling R7 en a besoin")
        self.sp_head_norm = norm

        self.sp_key = nn.Linear(d, 128)
        self.sp_pos = nn.Linear(16, 128, bias=False)
        nn.init.zeros_(self.sp_pos.weight)
        self.sp_q = nn.Parameter(torch.zeros(n_targets, 128))

        self.proj = nn.Sequential(nn.Linear(d, self.hidden), nn.LayerNorm(self.hidden))
        self.slot = nn.Embedding(10, self.hidden)
        self.view = nn.Embedding(6, self.hidden)
        self.geom = nn.Linear(4, self.hidden, bias=False)
        nn.init.zeros_(self.slot.weight)
        nn.init.zeros_(self.view.weight)
        nn.init.zeros_(self.geom.weight)
        self.drop = nn.Dropout(dropout)
        self.win_proj = nn.Linear(self.hidden, self.hidden)
        self.q_win = nn.Parameter(torch.randn(n_targets, self.hidden) * 0.02)
        self.out_norm = nn.LayerNorm(self.hidden)
        self.cls_w = nn.Parameter(torch.randn(n_targets, self.hidden) * 0.02)
        self.bias = nn.Parameter(torch.zeros(n_targets))

        self.ext_u = nn.Parameter(torch.zeros(n_targets, self.hidden))
        self.ext_b = nn.Parameter(torch.full((n_targets,), -4.0))
        self.ext_gamma = nn.Parameter(torch.zeros(n_targets))
        self.ext_view = nn.Parameter(torch.zeros(n_targets, 6))
        em = torch.zeros(n_targets)
        em[OA_MED] = em[OA_LAT] = em[OA_PF] = 1.0
        self.register_buffer("ext_mask", em)
        self.syn_beta = nn.Parameter(torch.zeros(()))

        self.register_buffer(
            "img_mean", torch.tensor(self.encoder.mean).view(1, 3, 1, 1),
            persistent=True,
        )
        self.register_buffer(
            "img_std", torch.tensor(self.encoder.std).view(1, 3, 1, 1),
            persistent=True,
        )
        self.register_buffer("slot_plane", torch.tensor(SLOT_PLANE, dtype=torch.long))
        self.register_buffer("slot_fluid", torch.tensor(SLOT_FLUID, dtype=torch.float32))
        print(
            f"KneeNet-CNX-r7 | {self.backbone_name} | hidden {self.hidden} "
            f"| spatial + fenetres + etendue OA + syn_beta "
            f"| slots {self.n_series} | dim {d}",
            flush=True,
        )

    def set_encoder_frozen(self, flag: bool) -> None:
        for p in self.encoder.parameters():
            p.requires_grad = not flag

    def encoder_parameters(self):
        yield from self.encoder.parameters()

    def head_parameters(self):
        for name, p in self.named_parameters():
            if name.startswith("encoder."):
                continue
            yield p

    @torch.no_grad()
    def extra_log(self) -> dict[str, float]:
        return {
            "syn_beta": float(self.syn_beta.detach().cpu()),
            "ext_gamma_abs": float(self.ext_gamma.detach().abs().mean()),
            "sp_q_abs": float(self.sp_q.detach().abs().mean()),
        }

    def _stem_rgb(self, x: torch.Tensor, idx: torch.Tensor) -> torch.Tensor:
        """Trois coupes adjacentes -> 3 canaux. x (B,S,D,H,W)."""
        _b, s, d, _h, _w = x.shape
        bs = idx // (s * d)
        rest = idx % (s * d)
        ss = rest // d
        dd = rest % d
        d_m = (dd - 1).clamp(0, d - 1)
        d_p = (dd + 1).clamp(0, d - 1)
        rgb = torch.stack(
            [x[bs, ss, d_m], x[bs, ss, dd], x[bs, ss, d_p]],
            dim=1,
        )
        return (rgb - self.img_mean) / self.img_std

    def _encode_indexed(self, x: torch.Tensor, idx: torch.Tensor) -> torch.Tensor:
        chunk = max(1, self.encode_chunk)
        outs = []
        for i in range(0, int(idx.numel()), chunk):
            rgb = self._stem_rgb(x, idx[i:i + chunk])
            if self.training:
                feats = checkpoint(self.encoder, rgb, use_reentrant=False)
            else:
                feats = self.encoder(rgb)
            outs.append(feats)
        return torch.cat(outs, dim=0)

    def _pos_enc(self, h: int, w: int, device: torch.device, dtype: torch.dtype) -> torch.Tensor:
        yy, xx = torch.meshgrid(
            torch.linspace(-1, 1, h, device=device),
            torch.linspace(-1, 1, w, device=device),
            indexing="ij",
        )
        fr = torch.tensor([1.0, 2.0, 4.0, 8.0], device=device) * (math.pi / 2)
        yy = yy.reshape(-1, 1) * fr
        xx = xx.reshape(-1, 1) * fr
        return torch.cat([yy.sin(), yy.cos(), xx.sin(), xx.cos()], 1).to(dtype)

    def spatial_pool(self, maps: torch.Tensor) -> torch.Tensor:
        """maps (N, d, h, w) -> (N, C, d). Requete nulle = moyenne des positions."""
        _n, _d, h, w = maps.shape
        f = maps.flatten(2).transpose(1, 2)
        pe = self._pos_enc(h, w, f.device, f.dtype)
        kin = F.layer_norm(f.float(), (f.shape[-1],)).to(f.dtype)
        k = torch.tanh(self.sp_key(kin) + self.sp_pos(pe)[None])
        att = torch.einsum("npk,ck->npc", k, self.sp_q.to(k.dtype)).float().softmax(1).to(f.dtype)
        g = torch.einsum("npc,npd->ncd", att, f)
        nrm = self.sp_head_norm
        g = F.layer_norm(g.float(), (g.shape[-1],), nrm.weight, nrm.bias, nrm.eps).to(f.dtype)
        return g

    def _window_meta(self, valid: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        """valid (B,S,D) -> mask (B,W), meta (B,W,9)."""
        b, s, d = valid.shape
        device = valid.device
        rel = torch.linspace(0, 1, d, device=device)
        n_valid = valid.flatten(2).sum(-1).clamp(min=1).to(torch.float32)
        slot = torch.arange(s, device=device)
        plane = self.slot_plane[:s].to(device)
        fluid = self.slot_fluid[:s].to(device)
        meta = valid.new_zeros(b, s, d, 9, dtype=torch.float32)
        meta[..., 0] = slot.view(1, s, 1).float()
        meta[..., 2] = rel.view(1, 1, d)
        meta[..., 3] = math.log1p(SLAB_MM)
        meta[..., 4] = 1.0
        meta[..., 5] = fluid.view(1, s, 1)
        meta[..., 6] = slot.view(1, s, 1).float()
        meta[..., 7] = plane.view(1, s, 1).float()
        meta[..., 8] = n_valid.view(b, s, 1)
        return valid.reshape(b, s * d), meta.reshape(b, s * d, 9)

    def extent_area(self, e: torch.Tensor, meta: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        """e (B,W,C) -> aire proxy (B,C), mm^2. Meme combinaison de plans que R7."""
        v = 6
        sid = meta[..., 6].long()
        s_max = max(16, int(sid.max()) + 1)
        vid = meta[..., 1].long().clamp(0, v - 1)
        q = torch.sigmoid(e.float()) * mask[..., None]
        ohv = F.one_hot(sid * v + vid, s_max * v).float() * mask[..., None]
        cntv = ohv.sum(1)
        mean_v = (
            torch.einsum("bwk,bwc->bkc", ohv, q) / cntv.clamp_min(1)[..., None]
        ).view(e.shape[0], s_max, v, -1)
        present = (cntv > 0).view(e.shape[0], s_max, v, 1)
        wv = self.ext_view.float().t()[None, None].masked_fill(~present, -1e4).softmax(2) * present
        mean = (wv * mean_v).sum(2)
        oh = F.one_hot(sid, s_max).float() * mask[..., None]
        cnt = oh.sum(1)
        span = torch.expm1(meta[..., 3].float()).clamp_min(0) * meta[..., 8].float()
        span_s = torch.einsum("bws,bw->bs", oh, span) / cnt.clamp_min(1)
        plane_s = (torch.einsum("bws,bw->bs", oh, meta[..., 7].float()) / cnt.clamp_min(1)).round()
        energy = mean * span_s[..., None]
        per_plane = []
        for pl in (0, 1, 2):
            ok = ((plane_s == pl) & (cnt > 0))[..., None]
            per_plane.append(energy.masked_fill(~ok, 0.0).amax(1))
        sag, cor, ax = per_plane
        area = torch.zeros_like(sag)
        area[:, OA_MED] = sag[:, OA_MED] * cor[:, OA_MED]
        area[:, OA_LAT] = sag[:, OA_LAT] * cor[:, OA_LAT]
        area[:, OA_PF] = ax[:, OA_PF] * sag[:, OA_PF]
        return area

    def _decompose_syn(self, z: torch.Tensor) -> torch.Tensor:
        ze = z[:, EFF].detach()
        onehot = z.new_zeros(z.shape[1])
        onehot[SYN] = 1
        return z + (self.syn_beta * ze)[:, None] * onehot[None, :]

    def head(self, tok: torch.Tensor, meta: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        """tok (N,C,d) des fenetres valides, dans l'ordre du masque. -> (B,C)."""
        b, w = mask.shape
        f = self.proj(tok.float())
        emb = (
            self.slot(meta[..., 0].long().clamp(0, 9))
            + self.view(meta[..., 1].long().clamp(0, 5))
            + self.geom(meta[..., 2:6])
        )
        t = f.new_zeros((b, w) + tuple(f.shape[1:]))
        t[mask] = f
        t = self.drop(t + emb[:, :, None, :])
        s2 = torch.einsum("bwch,ch->bwc", torch.tanh(self.win_proj(t)), self.q_win).float()
        s2 = s2.masked_fill(~mask[:, :, None], float("-inf"))
        a2 = s2.softmax(1).to(t.dtype)
        study = self.out_norm(torch.einsum("bwc,bwch->bch", a2, t))
        e_win = (t.float() * self.ext_u[None, None]).sum(-1) + self.ext_b
        z = (study * self.cls_w).sum(-1).float() + self.bias
        area = self.extent_area(e_win, meta, mask)
        z = z + self.ext_gamma * self.ext_mask * torch.log1p(area / 100.0)
        return self._decompose_syn(z)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x (B, S, D, H, W) dans [0, 1] -> logits (B, 12)."""
        _b, s, d, _h, _w = x.shape
        if s != self.n_series:
            raise ValueError(f"attendu {self.n_series} series, recu {s}")
        if d > self.max_depth:
            raise ValueError(f"D={d} > max_depth={self.max_depth}")
        valid = x.amax(dim=(-2, -1)) > 1e-6
        mask, meta = self._window_meta(valid)
        idx = mask.reshape(-1).nonzero(as_tuple=False).view(-1)
        if idx.numel() == 0:
            return x.new_zeros(x.shape[0], self.n_targets)
        maps = self._encode_indexed(x, idx)
        tok = self.spatial_pool(maps)
        return self.head(tok, meta, mask)


In [ ]:
import numpy as np
import pandas as pd

import torch

from pathlib import Path

TARGETS = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA',
           'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's",
           'Contusion', 'Fracture']

N_TARGETS = len(TARGETS)


@torch.no_grad()
def _predict_logits(model, x: torch.Tensor) -> torch.Tensor:
    # ConvNeXt depthwise : cuDNN T4 ne trouve pas d'algo en bf16/fp16.
    with torch.autocast("cuda", enabled=False):
        return model(x.float())


def load_model(weights_path: str | Path) -> tuple[torch.nn.Module, dict]:
    ckpt = torch.load(str(weights_path), map_location="cpu", weights_only=False)
    cfg = ckpt.get("config", {}) if isinstance(ckpt, dict) else {}
    if str(cfg.get("model")) != "kneenet_cnx_r7":
        raise RuntimeError(f"model inattendu {cfg.get('model')}")
    if str(cfg.get("run_name")) != "cnx_r7_in22k_v3b_oofmask_full":
        raise RuntimeError(f"run inattendu {cfg.get('run_name')}")
    if str(cfg.get("backbone")) != "convnext_small.fb_in22k_ft_in1k":
        raise RuntimeError(f"backbone inattendu {cfg.get('backbone')}")
    if cfg.get("laterality"):
        raise RuntimeError("laterality inattendue")
    if int(cfg.get("hidden", -1)) != 384:
        raise RuntimeError(f"hidden inattendu {cfg.get('hidden')}")
    if abs(float(cfg.get("dropout", -1.0)) - 0.1) > 1e-6:
        raise RuntimeError(f"dropout inattendu {cfg.get('dropout')}")
    if str(ckpt.get("epoch")) != "swa-7-8-9-10":
        raise RuntimeError(f"epoch inattendue {ckpt.get('epoch')}")
    if list(ckpt.get("swa_epochs") or []) != [7, 8, 9, 10]:
        raise RuntimeError(f"swa_epochs inattendu {ckpt.get('swa_epochs')}")
    n_series = int(cfg.get("n_series", 6))
    if n_series != 6:
        raise RuntimeError(f"n_series {n_series}")
    encode_chunk_cfg = int(cfg.get("encode_chunk", 16))
    encode_chunk = min(4, encode_chunk_cfg)
    if encode_chunk != encode_chunk_cfg:
        print(f"encode_chunk {encode_chunk_cfg} -> {encode_chunk} (T4 fp32)", flush=True)
    dropout = float(cfg.get("dropout", 0.1))
    max_depth = int(cfg.get("max_depth", 64) or 64)
    model = KneeNet(
        backbone=str(cfg.get("backbone")),
        n_series=n_series,
        n_targets=N_TARGETS,
        hidden=int(cfg.get("hidden", 384)),
        dropout=dropout,
        pretrained=False,
        encode_chunk=encode_chunk,
        max_depth=max_depth,
    )
    model.load_state_dict(ckpt["model"], strict=True)
    print("load strict OK", flush=True)
    gold = ckpt.get("gold_macro_auc")
    gold_txt = "n/a" if gold is None else round(float(gold), 4)
    print(
        "modele", cfg.get("model"), "| run", cfg.get("run_name"),
        "| epoch", ckpt.get("epoch"), "| gold in-sample", gold_txt,
        "| swa", ckpt.get("swa_epochs"),
        "| n_series", n_series, "| img 224 | crop", CROP_MM, "mm | TTA x3 | 2 GPU",
        flush=True,
    )
    model.eval()
    return model, cfg


test = pd.read_csv(TEST_CSV)
series = pd.read_csv(TEST_SERIES_CSV)
test_series_dir = Path(TEST_SERIES_DIR)

series_by_uid = {uid: g for uid, g in series.groupby("StudyInstanceUID", sort=False)}

import copy
from concurrent.futures import ThreadPoolExecutor

def _hflip(x):
    """Retourner l'image, sans rotation. Sagittal : axe D. Coronal/Axial : axe W."""
    if x.ndim != 5 or int(x.shape[1]) != 6:
        raise RuntimeError(tuple(x.shape))
    y = x.clone()
    for s in (0, 3):
        y[:, s] = torch.flip(y[:, s], dims=(1,))
    for s in (1, 2, 4, 5):
        y[:, s] = torch.flip(y[:, s], dims=(3,))
    return y


def _swap_medlat(p):
    q = p.clone()
    q[:, 2] = p[:, 3]
    q[:, 3] = p[:, 2]
    q[:, 4] = p[:, 5]
    q[:, 5] = p[:, 4]
    return q


def _center_zoom(x, scale):
    """Zoom avant par center-crop puis resize. Pas de rotation."""
    if x.ndim != 5 or float(scale) <= 1.0:
        raise RuntimeError((tuple(x.shape), scale))
    b, s, d, h, w = x.shape
    ch = int(round(h / float(scale)))
    cw = int(round(w / float(scale)))
    if ch < 8 or cw < 8 or ch >= h or cw >= w:
        raise RuntimeError((ch, cw, h, w, scale))
    y0, x0 = (h - ch) // 2, (w - cw) // 2
    crop = x[:, :, :, y0:y0 + ch, x0:x0 + cw]
    flat = crop.reshape(b * s * d, 1, ch, cw)
    up = torch.nn.functional.interpolate(flat, size=(h, w), mode="bilinear", align_corners=False)
    return up.reshape(b, s, d, h, w)


def _tta(model, device, x_cpu):
    with torch.no_grad():
        if str(device).startswith("cuda"):
            torch.cuda.set_device(device)
        x = x_cpu.to(device).float()
        with torch.autocast("cuda", enabled=False):
            p_flip = torch.sigmoid(model(_hflip(x)).float())
            p_z1 = torch.sigmoid(model(_center_zoom(x, 1.05)).float())
            p_z2 = torch.sigmoid(model(_center_zoom(x, 1.10)).float())
        return ((_swap_medlat(p_flip) + p_z1 + p_z2) / 3.0).float().cpu()


N_GPU = torch.cuda.device_count() if DEVICE == "cuda" else 1
if DEVICE == "cuda" and N_GPU < 2:
    raise RuntimeError(("attendu 2 GPU", N_GPU))
devices = ["cuda:0", "cuda:1"] if DEVICE == "cuda" else ["cpu"]
base_model, _cfg = load_model(WEIGHTS_PATH)
models = [copy.deepcopy(base_model).to(dev).eval() for dev in devices]
del base_model
n_series = int(getattr(models[0], "n_series", 6))
print(
    f"{len(test)} etudes | TTA x3 hflip + zoom 1.05 + zoom 1.10, "
    "swap seulement apres hflip, sans identite, pas de rotation | "
    f"gpus visibles {N_GPU} | {' '.join(devices)} | cnx forward fp32",
    flush=True,
)

pool = ThreadPoolExecutor(max_workers=len(devices))
inflight = []
rows_map = {}
uids = list(test.StudyInstanceUID)
for i, uid in enumerate(uids):
    study_dir = test_series_dir / uid
    srows = series_by_uid.get(uid, series.iloc[0:0])
    vol = study_to_tensor(
        study_dir,
        srows,
        n_series=n_series,
        rgb_neighbors=False,
        img_size=224,
    )
    if vol.shape != (n_series, 24, 224, 224):
        raise RuntimeError(f"volume inattendu {vol.shape}")
    x0 = torch.from_numpy(vol.copy()).unsqueeze(0)
    slot = i % len(devices)
    inflight.append((uid, devices[slot], pool.submit(_tta, models[slot], devices[slot], x0)))
    if len(inflight) >= len(devices):
        uid_d, dev_d, fut_d = inflight.pop(0)
        rows_map[uid_d] = fut_d.result().numpy()[0]
        if (i + 1) % 25 == 0 or len(uids) <= 20:
            print(f"{i+1}/{len(uids)} {dev_d}", flush=True)
while inflight:
    uid_d, dev_d, fut_d = inflight.pop(0)
    rows_map[uid_d] = fut_d.result().numpy()[0]
    print(f"fini {uid_d} {dev_d}", flush=True)
pool.shutdown(wait=True)
if set(map(str, rows_map)) != set(map(str, uids)):
    raise RuntimeError(("etudes manquantes", len(rows_map), len(uids)))
rows = [[uid] + rows_map[uid].tolist() for uid in uids]

pred = pd.DataFrame(rows, columns=["StudyInstanceUID"] + TARGETS)

sample = pd.read_csv(SAMPLE_SUB)
sub = pred.set_index("StudyInstanceUID").reindex(sample.StudyInstanceUID).reset_index()
sub = sub[list(sample.columns)]
prior = sub[TARGETS].mean()
sub[TARGETS] = sub[TARGETS].fillna(prior)

out_dir = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
out_csv = out_dir / "submission.csv"
sub.to_csv(out_csv, index=False)
cwd_csv = Path("submission.csv")
if cwd_csv.resolve() != out_csv.resolve():
    sub.to_csv(cwd_csv, index=False)

print("ecrit", out_csv.resolve(), "| lignes", len(sub), "| existe", out_csv.exists())
print(sub.shape, "| colonnes OK :", list(sub.columns) == list(sample.columns))
print(sub[TARGETS].describe().loc[["mean", "std", "min", "max"]].round(3).to_string())
assert out_csv.exists() and out_csv.stat().st_size > 0, "submission.csv manquant"
assert not sub[TARGETS].isna().any().any(), "NaN dans la submission !"
assert list(sub.columns) == list(sample.columns), "colonnes != sample_submission"
